# Household Waste Image Classification: FCNN vs CNN, both trained from scratch

**UECS2523 Applied Machine Learning, group assignment**
Benjamin Thio Zi Liang · Chew Zi Jing · Jeremy Gabriel Lee Ji Hin

Every model in this notebook starts from random weights. There are no pretrained weights and no transfer learning anywhere.

| Part | What happens | Owner |
|---|---|---|
| 1 | Setup: GPU check, Google Drive, settings | all |
| 2 | Download the dataset | Benjamin |
| 3 | Data audit: unreadable files and duplicates | Benjamin |
| 4 | Exploratory data analysis (EDA) | Benjamin |
| 5 | Train / validation / test split and preprocessing | Benjamin |
| 6 | FCNN experiments (from scratch) | Chew Zi Jing |
| 7 | CNN experiments (from scratch) | Jeremy |
| 8 | Test-set evaluation and comparison | all |
| 9 | Grad-CAM: where the CNN looks | Jeremy |
| 10 | Compute budget actually used | Chew Zi Jing |
| 11 | Export the best models for the real-life photo demo | Benjamin |
| 12 | Optional: repeat the cheaper runs with 2 more seeds | Chew Zi Jing |

### How to run it on Google Colab
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. `Runtime → Run all`. Colab asks for Google Drive access once; allow it so the trained models survive a disconnect.
3. Parts 1-11 take roughly 1 to 1.5 hours on a free T4; the optional Part 12 adds about 40 minutes. Set `QUICK_RUN = True` first if you only want a 5-minute check that everything works.
4. If Colab disconnects, run everything again. Finished experiments are loaded from Drive instead of being retrained.

## 1. Setup

In [ ]:
# ---- Settings you may want to change --------------------------------------
QUICK_RUN = False          # True = tiny subset + 2 epochs, just to check the pipeline end to end
USE_DRIVE = True           # save models/results to Google Drive (recommended on Colab)
SKIP_TRAINED = True        # reuse experiments already saved in the output folder
PROJECT_NAME = "AML_Waste_Project"

KAGGLE_DATASET = "mostafaabla/garbage-classification"   # 12 classes, ~15.5k images
IMG_SIZE = 128             # every image is stored once at 128x128; FCNN runs downsample inside the model
BATCH_SIZE = 64
SEED = 42
VAL_FRAC, TEST_FRAC = 0.15, 0.15
LABEL_SMOOTHING = 0.1
MAX_EPOCHS_FCNN = 80       # same cap for every run; early stopping usually ends runs sooner
MAX_EPOCHS_CNN = 80
RUN_EXTRA_SEEDS = True     # Part 12: repeat the cheaper runs with 2 more seeds (about 40 extra minutes)
EARLY_STOP_PATIENCE = 12
# ---------------------------------------------------------------------------

In [ ]:
import os, sys, io, json, time, hashlib, shutil, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore", category=UserWarning)
tf.get_logger().setLevel("ERROR")          # hide TensorFlow's retracing / deprecation chatter
Image.MAX_IMAGE_PIXELS = 200_000_000
RESAMPLE = getattr(Image, "Resampling", Image).LANCZOS
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False})

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Python {sys.version.split()[0]} | TensorFlow {tf.__version__} | Keras {keras.__version__} | Colab: {IN_COLAB}")

**GPU check.** Mixed precision (float16 maths with float32 weights) is switched on when a GPU is present. On a T4 it roughly halves training time and memory, which is part of our answer to the compute question.

In [ ]:
GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass

if GPUS:
    GPU_NAME = tf.config.experimental.get_device_details(GPUS[0]).get("device_name", "GPU")
    keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"GPU found: {GPU_NAME}. Mixed precision ON.")
else:
    GPU_NAME = "CPU only"
    print("No GPU found. Training will be very slow.")
    if IN_COLAB:
        print(">>> In Colab: Runtime > Change runtime type > T4 GPU, then run all cells again.")

keras.utils.set_random_seed(SEED)

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    OUT_DIR = Path.cwd() / PROJECT_NAME

if QUICK_RUN:
    OUT_DIR = OUT_DIR / "quick_run"
MODELS_DIR, LOGS_DIR, FIG_DIR, EXPORT_DIR = (OUT_DIR / d for d in ("models", "logs", "figures", "export"))
for d in (OUT_DIR, MODELS_DIR, LOGS_DIR, FIG_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Outputs go to:", OUT_DIR)


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

## 2. Download the dataset

The dataset is downloaded from Kaggle with `kagglehub`. Public datasets normally download without a login. If Kaggle asks for credentials, add two Colab secrets (key icon on the left): `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token), switch on notebook access, and run the cell again.

In [ ]:
DATASET_DIR_OVERRIDE = None   # e.g. "/content/drive/MyDrive/datasets/garbage_classification" if you already have a copy


def download_dataset():
    if DATASET_DIR_OVERRIDE:
        return Path(DATASET_DIR_OVERRIDE)
    try:
        import kagglehub
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q kagglehub")
        import kagglehub
    try:
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception as first_error:
        print("Anonymous download failed:", first_error)
        if IN_COLAB:
            from google.colab import userdata
            try:
                os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            except Exception as secret_error:
                raise RuntimeError("Add KAGGLE_USERNAME and KAGGLE_KEY as Colab secrets "
                                   "(key icon on the left) and run this cell again.") from secret_error
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))


IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}


def find_class_root(base):
    """Return the folder whose sub-folders hold the most images (one sub-folder per class)."""
    best, best_score = None, (0, 0)
    for dirpath, dirnames, _ in os.walk(base):
        n_cls = n_img = 0
        for d in dirnames:
            imgs = [f for f in os.listdir(os.path.join(dirpath, d)) if Path(f).suffix.lower() in IMG_EXTS]
            if imgs:
                n_cls, n_img = n_cls + 1, n_img + len(imgs)
        if n_cls >= 2 and (n_cls, n_img) > best_score:
            best, best_score = Path(dirpath), (n_cls, n_img)
    if best is None:
        raise FileNotFoundError(f"No class folders with images found under {base}")
    return best


DATA_BASE = download_dataset()
CLASS_ROOT = find_class_root(DATA_BASE)
CLASS_NAMES = sorted(d.name for d in CLASS_ROOT.iterdir()
                     if d.is_dir() and any(p.suffix.lower() in IMG_EXTS for p in d.iterdir()))
NUM_CLASSES = len(CLASS_NAMES)

# Collect files from every folder named like a class anywhere under the download
# (this also picks up train/test sub-splits or nested copies; exact copies are removed in Part 3).
files, labels = [], []
for dirpath, _, filenames in os.walk(DATA_BASE):
    cls = Path(dirpath).name
    if cls in CLASS_NAMES:
        for f in sorted(filenames):
            if Path(f).suffix.lower() in IMG_EXTS:
                files.append(os.path.join(dirpath, f))
                labels.append(cls)

print(f"Class root: {CLASS_ROOT}")
print(f"{NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"{len(files):,} image files found")

if QUICK_RUN:   # keep at most 60 images per class
    df_ = pd.DataFrame({"f": files, "l": labels})
    keep = pd.concat([g.sample(min(len(g), 60), random_state=SEED) for _, g in df_.groupby("l")])
    files, labels = keep["f"].tolist(), keep["l"].tolist()
    print(f"QUICK_RUN: using {len(files)} images")

## 3. Data audit: unreadable files and duplicates

Every image is opened once. In the same pass we:
* record format, colour mode, width, height and file size,
* fix phone-camera rotation (EXIF), put transparent PNGs on a white background and convert to RGB,
* compute an **MD5 hash** (byte-identical copies) and a **difference hash (dHash)** (the same picture re-saved or resized),
* centre-crop to a square and resize to 128 × 128, and keep the result in memory.

Centre-cropping keeps the object's shape. Padding would add borders, and the model could learn to use those borders to tell classes apart.

In [ ]:
def to_rgb(im):
    """Convert any PIL image to RGB; transparent areas become white."""
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.getchannel("A"))
        return bg
    return im.convert("RGB")


def center_crop_resize(im, size):
    w, h = im.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return im.crop((left, top, left + s, top + s)).resize((size, size), RESAMPLE)


def dhash(im, hash_size=8):
    g = np.asarray(im.convert("L").resize((hash_size + 1, hash_size), RESAMPLE), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten()).tobytes().hex()


def border_white_fraction(arr, frac=0.1):
    """Share of near-white pixels in the outer 10% border: a rough 'studio background' indicator."""
    s = arr.shape[0]
    b = max(1, int(s * frac))
    mask = np.ones((s, s), bool)
    mask[b:-b, b:-b] = False
    return float((arr[mask].min(axis=1) > 230).mean())


def read_image(path):
    raw = Path(path).read_bytes()
    info = {"md5": hashlib.md5(raw).hexdigest(), "kb": len(raw) / 1024}
    with Image.open(io.BytesIO(raw)) as im:
        info.update(fmt=im.format, mode=im.mode, width=im.size[0], height=im.size[1])
        im.load()
        im = to_rgb(ImageOps.exif_transpose(im))
    info["dhash"] = dhash(im)
    arr = np.asarray(center_crop_resize(im, IMG_SIZE), dtype=np.uint8)
    info["white_border"] = border_white_fraction(arr)
    info["brightness"] = float(arr.mean())
    return info, arr


X_all = np.zeros((len(files), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
records = [None] * len(files)


def _work(i):
    try:
        info, arr = read_image(files[i])
        X_all[i] = arr
        info["ok"], info["error"] = True, ""
    except Exception as e:  # unreadable / truncated file
        info = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    records[i] = info


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    list(tqdm(ex.map(_work, range(len(files))), total=len(files), desc="Reading images"))
print(f"Read {len(files):,} files in {time.perf_counter() - t0:.0f} s")

meta = pd.DataFrame(records)
meta.insert(0, "label", labels)
meta.insert(0, "path", [str(Path(f).relative_to(DATA_BASE)) for f in files])
meta["idx"] = np.arange(len(meta))

In [ ]:
audit = {"files found": len(meta)}

bad = meta[~meta["ok"]]
audit["unreadable files removed"] = len(bad)
if len(bad):
    display(bad[["path", "label", "error"]].head(10))
meta = meta[meta["ok"]].copy()

# Byte-identical copies filed under DIFFERENT classes: the label is ambiguous, so drop every copy.
md5_labels = meta.groupby("md5")["label"].nunique()
conflict_md5 = set(md5_labels[md5_labels > 1].index)
audit["images in cross-class duplicate groups (all removed)"] = int(meta["md5"].isin(conflict_md5).sum())
meta = meta[~meta["md5"].isin(conflict_md5)]

# Byte-identical copies inside the same class: keep the first one.
dup_mask = meta.duplicated("md5", keep="first")
audit["exact duplicates removed (same class)"] = int(dup_mask.sum())
dups_per_class = meta[dup_mask]["label"].value_counts()
meta = meta[~dup_mask].reset_index(drop=True)

# Near-duplicates (re-saved or resized copies): difference hashes at most 4 bits apart AND near-identical
# 32x32 thumbnails (small mean difference AND correlation above 0.95). The thumbnail checks throw out hash
# collisions: different objects on plain white backgrounds can share a hash and a similar mean colour.
# Near-duplicates are kept but locked into one split (Part 5).
bits = np.unpackbits(np.frombuffer(bytes.fromhex("".join(meta["dhash"])), np.uint8).reshape(-1, 8), axis=1).astype(np.float32)
ones = bits.sum(1)
k = IMG_SIZE // 32
thumb = np.concatenate([X_all[meta["idx"].values[s:s + 2000]].reshape(-1, 32, k, 32, k, 3).mean(axis=(2, 4, 5))
                        for s in range(0, len(meta), 2000)]).astype(np.float32)
tz = thumb.reshape(len(thumb), -1)
tz = (tz - tz.mean(1, keepdims=True)) / (tz.std(1, keepdims=True) + 1e-6)   # for the correlation check
parent = np.arange(len(meta))


def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a


for s in range(0, len(meta), 2000):
    hamming = ones[s:s + 2000, None] + ones[None, :] - 2 * bits[s:s + 2000] @ bits.T
    for i, j in zip(*np.nonzero(hamming <= 4)):
        i += s
        if i < j and np.abs(thumb[i] - thumb[j]).mean() < 8 and float(tz[i] @ tz[j]) / tz.shape[1] > 0.95:
            parent[find(i)] = find(j)
meta["group"] = [find(i) for i in range(len(meta))]
gsize = meta.groupby("group")["idx"].transform("size")
audit["images with a near-duplicate (kept, same split)"] = int((gsize > 1).sum())
audit["near-duplicate groups"] = int(meta.loc[gsize > 1, "group"].nunique())
audit["near-duplicate groups spanning 2+ classes"] = int((meta.groupby("group")["label"].nunique() > 1).sum())
audit["images after cleaning"] = len(meta)

audit_df = pd.DataFrame.from_dict(audit, orient="index", columns=["count"])
audit_df.to_csv(LOGS_DIR / "data_audit.csv")
display(audit_df)
if len(dups_per_class):
    print("Exact duplicates removed per class:")
    display(dups_per_class.to_frame("duplicates removed"))

## 4. Exploratory data analysis

In [ ]:
counts = meta["label"].value_counts().reindex(CLASS_NAMES)
imbalance = counts.max() / counts.min()

fig, ax = plt.subplots(figsize=(10, 3.6))
order = counts.sort_values(ascending=False)
bars = ax.bar(order.index, order.values, color="#4C78A8")
ax.bar_label(bars, fontsize=8)
ax.set_ylabel("images")
ax.set_title(f"Images per class after cleaning (largest / smallest = {imbalance:.1f}x)")
plt.xticks(rotation=35, ha="right")
save_fig(fig, "eda_class_counts"); plt.show()

In [ ]:
n_show = 6
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(NUM_CLASSES, n_show, figsize=(n_show * 1.4, NUM_CLASSES * 1.45))
for r, cls in enumerate(CLASS_NAMES):
    idxs = rng.choice(meta.loc[meta["label"] == cls, "idx"].values, n_show, replace=False)
    for c, i in enumerate(idxs):
        ax = axes[r, c]
        ax.imshow(X_all[i]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c == 0:
            ax.set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle("Random samples per class (after centre crop + resize)", y=1.0)
plt.tight_layout(); save_fig(fig, "eda_samples"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].scatter(meta["width"], meta["height"], s=3, alpha=0.3)
axes[0].set(xscale="log", yscale="log", xlabel="width (px)", ylabel="height (px)", title="Original image sizes")
(meta["width"] / meta["height"]).clip(0.25, 4).plot.hist(bins=60, ax=axes[1])
axes[1].set(xlabel="width / height", title="Aspect ratio")
meta.boxplot(column="kb", by="label", ax=axes[2], rot=60, showfliers=False)
axes[2].set(title="File size by class (KB)", xlabel=""); fig.suptitle("")
plt.tight_layout(); save_fig(fig, "eda_sizes"); plt.show()

display(pd.crosstab(meta["label"], meta["fmt"].fillna("?") + "/" + meta["mode"].fillna("?")))
print(meta[["width", "height", "kb"]].describe().round(1))

**Background bias check.** If one class is mostly shot on white studio backgrounds, a model can learn "white background → that class" instead of the object itself. Such a shortcut breaks on real-life photos. We measure it here, test whether the trained models use it in Part 8, and test it again with our own photos on white and dark surfaces.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
meta.boxplot(column="white_border", by="label", ax=axes[0], rot=60, showfliers=False)
axes[0].set(title="Share of near-white pixels in the image border", xlabel="")
meta.boxplot(column="brightness", by="label", ax=axes[1], rot=60, showfliers=False)
axes[1].set(title="Mean brightness (0-255)", xlabel=""); fig.suptitle("")
plt.tight_layout(); save_fig(fig, "eda_background_bias"); plt.show()

bg = meta.groupby("label")["white_border"].agg(lambda s: (s > 0.6).mean()).rename("share with mostly-white border")
display(bg.sort_values(ascending=False).to_frame().style.format("{:.0%}"))

In [ ]:
fig, axes = plt.subplots(2, (NUM_CLASSES + 1) // 2, figsize=(2 * ((NUM_CLASSES + 1) // 2), 4.4))
for ax, cls in zip(axes.flat, CLASS_NAMES):
    ax.imshow(X_all[meta.loc[meta["label"] == cls, "idx"].values].mean(axis=0, dtype=np.float32).astype(np.uint8))
    ax.set_title(cls, fontsize=9); ax.axis("off")
for ax in list(axes.flat)[NUM_CLASSES:]:
    ax.axis("off")
fig.suptitle("Mean image per class: a centred blob means objects sit in the middle; a pale frame means white backgrounds")
plt.tight_layout(); save_fig(fig, "eda_mean_images"); plt.show()

**Can raw pixels separate the classes?** An FCNN sees the image as a flat vector of pixels. PCA on those vectors shows how much the classes overlap before any learning, which is what the FCNN has to work with.

In [ ]:
sub = pd.concat([g.sample(min(len(g), 250), random_state=SEED) for _, g in meta.groupby("label")])
small = tf.image.resize(X_all[sub["idx"].values].astype("float32"), (32, 32)).numpy().reshape(len(sub), -1) / 255.0
pca = PCA(n_components=50, random_state=SEED).fit(small)
Z = pca.transform(small)[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cmap = plt.get_cmap("tab20", NUM_CLASSES)
for k, cls in enumerate(CLASS_NAMES):
    m = (sub["label"] == cls).values
    axes[0].scatter(Z[m, 0], Z[m, 1], s=6, alpha=0.6, color=cmap(k), label=cls)
axes[0].legend(markerscale=3, fontsize=7, ncol=2); axes[0].set(title="Raw pixels projected on the first 2 principal components", xlabel="PC1", ylabel="PC2")
axes[1].plot(np.arange(1, 51), np.cumsum(pca.explained_variance_ratio_), marker=".")
axes[1].set(title="Cumulative explained variance", xlabel="components", ylabel="share of variance", ylim=(0, 1))
plt.tight_layout(); save_fig(fig, "eda_pca"); plt.show()

## 5. Split and preprocessing

* **Split 70 / 15 / 15**, stratified by class so every split keeps the class mix. Near-duplicate groups (Part 3) are kept together, so no picture appears in both training and test.
* **Test set** is opened once, at the end, for four models fixed in advance (Part 8). Every design decision uses the validation set.
* **Standardisation**: per-channel mean and standard deviation from the training images only, built into the model as a `Normalization` layer. A raw photo can then be fed straight in during the demo.
* **Class weights** (`balanced`) offset the class imbalance found in the EDA.

In [ ]:
group_label = meta.groupby("group")["label"].agg(lambda s: s.mode().iloc[0])
g_trainval, g_test = train_test_split(group_label.index.values, test_size=TEST_FRAC,
                                      stratify=group_label.values, random_state=SEED)
gl_tv = group_label.loc[g_trainval]
g_train, g_val = train_test_split(gl_tv.index.values, test_size=VAL_FRAC / (1 - TEST_FRAC),
                                  stratify=gl_tv.values, random_state=SEED)
split_of = {**{g: "train" for g in g_train}, **{g: "val" for g in g_val}, **{g: "test" for g in g_test}}
meta["split"] = meta["group"].map(split_of)

class_to_id = {c: i for i, c in enumerate(CLASS_NAMES)}
meta["y"] = meta["label"].map(class_to_id)
meta.to_csv(LOGS_DIR / "split.csv", index=False)

parts = {s: meta[meta["split"] == s] for s in ("train", "val", "test")}
X = {s: X_all[p["idx"].values] for s, p in parts.items()}
Y = {s: p["y"].values.astype("int32") for s, p in parts.items()}
del X_all  # free memory

split_table = pd.crosstab(meta["label"], meta["split"])[["train", "val", "test"]]
split_table.loc["TOTAL"] = split_table.sum()
display(split_table)

In [ ]:
def channel_stats(x, chunk=2048):
    s = np.zeros(3); ss = np.zeros(3); n = 0
    for i in range(0, len(x), chunk):
        b = x[i:i + chunk].astype(np.float64).reshape(-1, 3)
        s += b.sum(0); ss += (b ** 2).sum(0); n += len(b)
    mean = s / n
    return mean, ss / n - mean ** 2


MEAN, VAR = channel_stats(X["train"])
print("Training-set channel mean:", MEAN.round(2), " std:", np.sqrt(VAR).round(2))

CLASS_WEIGHT = dict(enumerate(compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=Y["train"])))
display(pd.Series({CLASS_NAMES[k]: v for k, v in CLASS_WEIGHT.items()}, name="class weight").round(2).to_frame().T)

AUTOTUNE = tf.data.AUTOTUNE


def make_ds(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, keras.utils.to_categorical(y, NUM_CLASSES)))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    return (ds.batch(BATCH_SIZE)
              .map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE)
              .prefetch(AUTOTUNE))


train_ds, val_ds, test_ds = make_ds(X["train"], Y["train"], True), make_ds(X["val"], Y["val"], False), make_ds(X["test"], Y["test"], False)

### Shared building blocks
* **Augmentation** (only active during training, runs on the GPU): random flips, small rotations, zoom, shifts, contrast and brightness changes. Phone photos differ from the dataset in exactly these ways.
* **Input block**: optional augmentation, optional downsampling (FCNN runs), then standardisation.

In [ ]:
def augmentation_block():
    return keras.Sequential([
        layers.RandomFlip("horizontal", dtype="float32"),
        layers.RandomRotation(0.05, fill_mode="reflect", dtype="float32"),       # about ±18 degrees
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect", dtype="float32"),
        layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", dtype="float32"),
        layers.RandomContrast(0.2, dtype="float32"),
        layers.RandomBrightness(0.15, value_range=(0, 255), dtype="float32"),
    ], name="augmentation")


def resizing_layer(size):
    # 128 -> 64 or 32 by averaging 2x2 or 4x4 pixel blocks: an exact "area" downsample.
    # (Resizing(antialias=True) would crash on a Colab GPU because XLA cannot compile it.)
    return layers.AveragePooling2D(pool_size=IMG_SIZE // size, dtype="float32", name=f"downsample_{size}")


def input_block(input_size, augment):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")   # raw RGB values 0-255
    x = augmentation_block()(inp) if augment else inp
    if input_size != IMG_SIZE:
        x = resizing_layer(input_size)(x)
    x = layers.Normalization(mean=MEAN.tolist(), variance=VAR.tolist(), dtype="float32", name="standardise")(x)
    return inp, x

## 6. FCNN experiments (trained from scratch)

A fully-connected network has to flatten the image, so a 64 × 64 × 3 input becomes 12,288 numbers. Each hidden unit gets its own weight for every pixel, so the parameter count grows with resolution and nothing is shared between image positions. Training an FCNN from scratch on this data needs care, so we:
* use **He initialisation** to suit ReLU,
* add **Batch Normalisation** to keep activations stable in a deep dense stack,
* use **Dropout + L2** because there are millions of weights and only ~10k training images,
* test **input resolution** (32 / 64 / 128), since a bigger input means many more weights.

The first four runs form a 2 × 2 design, so the effect of network size and the effect of regularisation can be read separately.

| Run | Hidden layers | Input | BN | Dropout | L2 | Augmentation |
|---|---|---|---|---|---|---|
| FCNN-A | 512-256 | 64 | no | 0.3 | 0 | no |
| FCNN-A-reg | 512-256 | 64 | yes | 0.4 | 1e-4 | no |
| FCNN-B-plain | 1024-512-256 | 64 | no | 0.3 | 0 | no |
| FCNN-B | 1024-512-256 | 64 | yes | 0.4 | 1e-4 | no |
| FCNN-B32 | 1024-512-256 | 32 | yes | 0.4 | 1e-4 | no |
| FCNN-B128 | 1024-512-256 | 128 | yes | 0.4 | 1e-4 | no |
| FCNN-C | 1024-512-256 | 64 | yes | 0.4 | 1e-4 | yes |

In [ ]:
def build_fcnn(name, input_size=64, hidden=(1024, 512, 256), dropout=0.4, batchnorm=True, l2=1e-4, augment=False):
    inp, x = input_block(input_size, augment)
    x = layers.Flatten(name="flatten")(x)
    reg = keras.regularizers.l2(l2) if l2 else None
    for i, units in enumerate(hidden, 1):
        x = layers.Dense(units, use_bias=not batchnorm, kernel_initializer="he_normal",
                         kernel_regularizer=reg, name=f"dense{i}")(x)
        if batchnorm:
            x = layers.BatchNormalization(name=f"bn{i}")(x)
        x = layers.Activation("relu", name=f"relu{i}")(x)
        if dropout:
            x = layers.Dropout(dropout, name=f"dropout{i}")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_probs")(x)
    return keras.Model(inp, out, name=name)


FCNN_RUNS = {
    "FCNN-A":       dict(input_size=64, hidden=(512, 256), dropout=0.3, batchnorm=False, l2=0.0),
    "FCNN-A-reg":   dict(input_size=64, hidden=(512, 256)),
    "FCNN-B-plain": dict(input_size=64, dropout=0.3, batchnorm=False, l2=0.0),
    "FCNN-B":       dict(input_size=64),
    "FCNN-B32":  dict(input_size=32),
    "FCNN-B128": dict(input_size=128),
    "FCNN-C":    dict(input_size=64, augment=True),
}
for n, kw in FCNN_RUNS.items():
    m = build_fcnn(n, **kw)
    print(f"{n:12s} parameters: {m.count_params():>12,}")
build_fcnn("FCNN-B", **FCNN_RUNS["FCNN-B"]).summary()

### Training loop shared by every experiment
* Loss: categorical cross-entropy with label smoothing 0.1, a mild regulariser that stops outputs being pushed to exactly 0 or 1. With 12 classes the target for the true class becomes 0.908.
* Class weights (`balanced`) in the training loss.
* Optimiser: Adam, learning rate 1e-3, halved when validation loss stalls for 4 epochs.
* Early stopping on **validation macro F1** (our main metric) after 12 epochs without improvement; the best weights are restored.
* Timing, the dtype policy actually used and each run's own peak GPU memory are logged for the compute-budget section.

In [ ]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t0)


BUILDERS = {}   # run name -> function that rebuilds the model
RESULTS = {}    # run name -> training record


def run_experiment(name, builder, max_epochs, lr=1e-3, seed=SEED):
    BUILDERS[name] = builder
    w_path, rec_path = MODELS_DIR / f"{name}.weights.h5", LOGS_DIR / f"{name}_record.json"
    if SKIP_TRAINED and w_path.exists() and rec_path.exists():
        model = builder()
        model.load_weights(w_path)
        model.compile(jit_compile=False)
        RESULTS[name] = json.loads(rec_path.read_text())
        print(f"[{name}] loaded from {w_path.name} (already trained)")
        return model

    keras.backend.clear_session()               # in Keras 3 this also resets the dtype policy to float32...
    if GPUS:
        keras.mixed_precision.set_global_policy("mixed_float16")   # ...so switch mixed precision back on
    keras.utils.set_random_seed(seed)
    gpu_base = 0
    if GPUS:  # measure this run's own peak, not memory held by earlier models
        tf.config.experimental.reset_memory_stats("GPU:0")
        gpu_base = tf.config.experimental.get_memory_info("GPU:0")["current"]
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=["accuracy", keras.metrics.F1Score(average="macro", name="macro_f1")],
                  jit_compile=False)   # same (non-XLA) code path for every run, so sec/epoch is comparable
    timer = EpochTimer()
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=EARLY_STOP_PATIENCE,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
        keras.callbacks.CSVLogger(str(LOGS_DIR / f"{name}_log.csv")),
        timer,
    ]
    epochs = 2 if QUICK_RUN else max_epochs
    print(f"\n=== Training {name} ({model.count_params():,} parameters, up to {epochs} epochs) ===")
    t0 = time.perf_counter()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=CLASS_WEIGHT,
                     callbacks=callbacks, verbose=2)
    total = time.perf_counter() - t0

    h = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    record = {
        "name": name,
        "seed": seed,
        "dtype_policy": model.dtype_policy.name,
        "params": int(model.count_params()),
        "epochs_run": len(h["loss"]),
        "hit_epoch_cap": len(h["loss"]) == epochs,
        "best_epoch": int(np.argmax(h["val_macro_f1"])) + 1,
        "val_macro_f1": float(max(h["val_macro_f1"])),
        "sec_per_epoch": float(np.median(timer.times[1:] if len(timer.times) > 1 else timer.times)),
        "train_minutes": total / 60,
        "peak_gpu_mb": (tf.config.experimental.get_memory_info("GPU:0")["peak"] - gpu_base) / 2**20 if GPUS else None,
        "history": h,
    }
    model.save_weights(w_path)
    rec_path.write_text(json.dumps(record))
    RESULTS[name] = record
    print(f"[{name}] done in {total/60:.1f} min ({record['dtype_policy']}), best epoch {record['best_epoch']}, "
          f"best val macro F1 {record['val_macro_f1']:.3f}")
    return model


MODELS = {}
for name, kw in FCNN_RUNS.items():
    MODELS[name] = run_experiment(name, lambda name=name, kw=kw: build_fcnn(name, **kw), MAX_EPOCHS_FCNN)

## 7. CNN experiments (trained from scratch)

The CNN keeps the image as a 2-D grid. A 3 × 3 filter is reused at every position (weight sharing), so it learns edges, textures and shapes once and spots them anywhere in the image. Pooling adds tolerance to small shifts.

Architecture: 4 blocks of [Conv 3×3 → BN → ReLU] × 2 + MaxPool, with 32 → 64 → 128 → 256 filters, then Global Average Pooling → Dropout 0.4 → Dense 256 → BN → ReLU → Dropout 0.4 → Softmax, with L2 1e-4 on every convolution. Global average pooling replaces a large Flatten + Dense layer and cuts parameters sharply.

| Run | Input | Augmentation | Purpose |
|---|---|---|---|
| CNN-A | 128 | no | effect of augmentation (compare with CNN-B) |
| CNN-B | 128 | yes | **main model** |
| CNN-B64 | 64 | yes | matched pair with FCNN-C: same 64-px input and augmentation, only the architecture differs |

In [ ]:
def conv_bn_relu(x, filters, name, l2):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=keras.regularizers.l2(l2) if l2 else None, name=f"{name}_conv")(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    return layers.Activation("relu", name=f"{name}_relu")(x)


def build_cnn(name, input_size=128, filters=(32, 64, 128, 256), block_dropout=0.1, head_dropout=0.4,
              l2=1e-4, augment=True):
    inp, x = input_block(input_size, augment)
    for b, f in enumerate(filters, 1):
        x = conv_bn_relu(x, f, f"block{b}a", l2)
        x = conv_bn_relu(x, f, f"block{b}b", l2)     # the last of these is the Grad-CAM layer
        x = layers.MaxPooling2D(2, name=f"block{b}_pool")(x)
        x = layers.Dropout(block_dropout, name=f"block{b}_dropout")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(head_dropout, name="head_dropout1")(x)
    x = layers.Dense(256, use_bias=False, kernel_initializer="he_normal", name="head_dense")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Activation("relu", name="head_relu")(x)
    x = layers.Dropout(head_dropout, name="head_dropout2")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_probs")(x)
    return keras.Model(inp, out, name=name)


GRADCAM_LAYER = "block4b_relu"
CNN_RUNS = {
    "CNN-A":   dict(input_size=128, augment=False),
    "CNN-B":   dict(input_size=128, augment=True),
    "CNN-B64": dict(input_size=64, augment=True),
}
for n, kw in CNN_RUNS.items():
    print(f"{n:8s} parameters: {build_cnn(n, **kw).count_params():>10,}")
build_cnn("CNN-B", **CNN_RUNS["CNN-B"]).summary()

In [ ]:
for name, kw in CNN_RUNS.items():
    MODELS[name] = run_experiment(name, lambda name=name, kw=kw: build_cnn(name, **kw), MAX_EPOCHS_CNN)

## 8. Evaluation and comparison

Metrics:
* **Macro F1** (main metric): F1 per class, averaged with equal weight, so small classes such as batteries count as much as clothes.
* **Balanced accuracy** (equal to macro recall): mean recall per class.
* **Accuracy**: share of correct predictions. Easy to read, but flattering when classes are imbalanced (clothes alone is about a third of the data).
* **Per-class precision / recall / F1**, **confusion matrices** and **bin-level accuracy** (does the item end up in the right recycling bin?).

**How the test set is used.** Every design decision (which FCNN, which CNN, does size, regularisation, resolution or augmentation help) is made on the **validation** set. The test set is opened once, for four models fixed in advance: the best FCNN, the best CNN, and the matched pair **FCNN-C vs CNN-B64** (same 64-px input and same augmentation, so only the architecture differs).

In [ ]:
def predict_probs(model, x):
    return model.predict(make_ds(x, np.zeros(len(x), "int32"), False), verbose=0)


def scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"macro_f1": f, "balanced_acc": balanced_accuracy_score(y_true, y_pred),   # balanced accuracy = macro recall
            "accuracy": accuracy_score(y_true, y_pred), "macro_precision": p}


PROBS, rows = {}, []
for name, model in MODELS.items():
    PROBS[name] = {"val": predict_probs(model, X["val"])}
    rec = RESULTS[name]
    row = {"run": name, "family": name.split("-")[0], "params": rec["params"]}
    row.update({f"val_{k}": v for k, v in scores(Y["val"], PROBS[name]["val"].argmax(1)).items()})
    row.update({"epochs": rec["epochs_run"], "cap hit": rec.get("hit_epoch_cap", False),
                "sec/epoch": rec["sec_per_epoch"], "train_min": rec["train_minutes"]})
    rows.append(row)
results_df = pd.DataFrame(rows).set_index("run")

BEST_FCNN = results_df[results_df["family"] == "FCNN"]["val_macro_f1"].idxmax()
BEST_CNN = results_df[results_df["family"] == "CNN"]["val_macro_f1"].idxmax()
MATCHED_PAIR = ("FCNN-C", "CNN-B64")
TEST_RUNS = list(dict.fromkeys([BEST_FCNN, BEST_CNN, *MATCHED_PAIR]))
for name in TEST_RUNS:                         # the only time the test set is used
    PROBS[name]["test"] = predict_probs(MODELS[name], X["test"])
    for k, v in scores(Y["test"], PROBS[name]["test"].argmax(1)).items():
        results_df.loc[name, f"test_{k}"] = v

results_df.to_csv(LOGS_DIR / "results_all_runs.csv")
fmt = {c: "{:.3f}" for c in results_df.columns if c.startswith(("val_", "test_"))}
fmt.update({"params": "{:,}", "sec/epoch": "{:.1f}", "train_min": "{:.1f}"})
display(results_df.style.format(fmt, na_rep="-").background_gradient(subset=["val_macro_f1"], cmap="Greens"))
print(f"Best FCNN (by validation macro F1): {BEST_FCNN}")
print(f"Best CNN  (by validation macro F1): {BEST_CNN}")
print(f"Test set opened for: {', '.join(TEST_RUNS)}")

**What each design choice did (validation macro F1).** Each row changes one thing. Part 12 repeats the cheaper runs with other seeds, which shows how big a difference has to be before it means anything.

In [ ]:
DESIGN = [
    ("FCNN size (512-256 -> 1024-512-256), no regularisation", "FCNN-A", "FCNN-B-plain"),
    ("FCNN size, with BN + Dropout + L2", "FCNN-A-reg", "FCNN-B"),
    ("Regularisation on the small FCNN", "FCNN-A", "FCNN-A-reg"),
    ("Regularisation on the large FCNN", "FCNN-B-plain", "FCNN-B"),
    ("FCNN input 32 -> 64 px", "FCNN-B32", "FCNN-B"),
    ("FCNN input 64 -> 128 px", "FCNN-B", "FCNN-B128"),
    ("Augmentation on the FCNN", "FCNN-B", "FCNN-C"),
    ("Augmentation on the CNN", "CNN-A", "CNN-B"),
    ("CNN input 64 -> 128 px", "CNN-B64", "CNN-B"),
    ("FCNN -> CNN at equal input and augmentation", "FCNN-C", "CNN-B64"),
]
design_df = pd.DataFrame([d for d in DESIGN if d[1] in results_df.index and d[2] in results_df.index],
                         columns=["question", "from", "to"])
design_df["val macro F1 (from)"] = design_df["from"].map(results_df["val_macro_f1"])
design_df["val macro F1 (to)"] = design_df["to"].map(results_df["val_macro_f1"])
design_df["change"] = design_df["val macro F1 (to)"] - design_df["val macro F1 (from)"]
design_df["params (from -> to)"] = [f"{results_df.loc[a, 'params'] / 1e6:.2f}M -> {results_df.loc[b, 'params'] / 1e6:.2f}M"
                                    for a, b in zip(design_df["from"], design_df["to"])]
design_df.to_csv(LOGS_DIR / "design_choices_validation.csv", index=False)
display(design_df.style.format({"val macro F1 (from)": "{:.3f}", "val macro F1 (to)": "{:.3f}", "change": "{:+.3f}"}))

In [ ]:
def plot_histories(names, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for n in names:
        h = RESULTS[n]["history"]
        line, = axes[0].plot(h["val_loss"], label=n)
        axes[0].plot(h["loss"], ls="--", color=line.get_color(), alpha=0.6)
        line, = axes[1].plot(h["val_macro_f1"], label=n)
        axes[1].plot(h["macro_f1"], ls="--", color=line.get_color(), alpha=0.6)
    axes[0].set(title="Loss (solid = validation, dashed = training)", xlabel="epoch")
    axes[1].set(title="Macro F1 (solid = validation, dashed = training)", xlabel="epoch")
    axes[1].legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); save_fig(fig, fname); plt.show()


plot_histories(list(FCNN_RUNS), "FCNN runs: learning curves", "curves_fcnn")
plot_histories(list(CNN_RUNS), "CNN runs: learning curves", "curves_cnn")

In [ ]:
def plot_confusions(names, fname):
    fig, axes = plt.subplots(1, len(names), figsize=(8 * len(names), 7))
    for ax, n in zip(np.atleast_1d(axes), names):
        cm = confusion_matrix(Y["test"], PROBS[n]["test"].argmax(1), labels=range(NUM_CLASSES), normalize="true")
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                if cm[i, j] >= 0.01:
                    ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                            color="white" if cm[i, j] > 0.5 else "black")
        ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES, fontsize=8)
        ax.set(xlabel="predicted", ylabel="true",
               title=f"{n}: test confusion matrix (row-normalised)\nmacro F1 = {results_df.loc[n, 'test_macro_f1']:.3f}")
    plt.tight_layout(); save_fig(fig, fname); plt.show()


plot_confusions([BEST_FCNN, BEST_CNN], "confusion_best_fcnn_vs_cnn")

In [ ]:
per_class = {}
for n in (BEST_FCNN, BEST_CNN):
    p, r, f, s = precision_recall_fscore_support(Y["test"], PROBS[n]["test"].argmax(1), labels=range(NUM_CLASSES), zero_division=0)
    per_class[n] = pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": s}, index=CLASS_NAMES)
per_class_df = pd.concat(per_class, axis=1)
per_class_df.to_csv(LOGS_DIR / "per_class_best_models.csv")
display(per_class_df.style.format("{:.3f}").format("{:.0f}", subset=[(n, "support") for n in per_class]))

fig, ax = plt.subplots(figsize=(12, 3.8))
w = 0.4
xpos = np.arange(NUM_CLASSES)
ax.bar(xpos - w / 2, per_class[BEST_FCNN]["f1"], w, label=BEST_FCNN, color="#E45756")
ax.bar(xpos + w / 2, per_class[BEST_CNN]["f1"], w, label=BEST_CNN, color="#4C78A8")
ax.set_xticks(xpos, CLASS_NAMES, rotation=35, ha="right"); ax.set_ylim(0, 1); ax.set_ylabel("F1 (test)")
ax.set_title("Per-class F1 on the test set"); ax.legend()
plt.tight_layout(); save_fig(fig, "per_class_f1"); plt.show()

print(classification_report(Y["test"], PROBS[BEST_CNN]["test"].argmax(1), target_names=CLASS_NAMES, digits=3))

**Most-confused class pairs.** Off-diagonal counts, largest first. These usually make physical sense (the glass colours, plastic vs. glass bottles, paper vs. cardboard).

In [ ]:
def top_confusions(name, k=8):
    cm = confusion_matrix(Y["test"], PROBS[name]["test"].argmax(1), labels=range(NUM_CLASSES))
    np.fill_diagonal(cm, 0)
    pairs = [(CLASS_NAMES[i], CLASS_NAMES[j], cm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if cm[i, j] > 0]
    return pd.DataFrame(sorted(pairs, key=lambda t: -t[2])[:k], columns=["true", "predicted", "count"])


display(pd.concat({BEST_FCNN: top_confusions(BEST_FCNN), BEST_CNN: top_confusions(BEST_CNN)}, axis=1))

**Right bin, wrong class?** Mixing up white glass and green glass costs nothing at home, because all glass goes in the brown bin. Mistaking a battery for metal does. So we also map the 12 classes to what a household does with them (Malaysia's blue / brown / orange recycling bins plus separate collection) and score that.

In [ ]:
BIN = {"paper": "blue bin", "cardboard": "blue bin",
       "brown-glass": "brown bin", "green-glass": "brown bin", "white-glass": "brown bin", "glass": "brown bin",
       "plastic": "orange bin", "metal": "orange bin",
       "battery": "e-waste collection", "clothes": "textile donation", "shoes": "textile donation",
       "biological": "food waste", "trash": "general waste"}
if all(c in BIN for c in CLASS_NAMES):
    bin_of = np.array([BIN[c] for c in CLASS_NAMES])
    bin_rows = []
    for n in TEST_RUNS:
        pred = PROBS[n]["test"].argmax(1)
        bin_rows.append({"run": n, "12-class accuracy": accuracy_score(Y["test"], pred),
                         "bin accuracy": float(np.mean(bin_of[pred] == bin_of[Y["test"]])),
                         "bin balanced accuracy": balanced_accuracy_score(bin_of[Y["test"]], bin_of[pred])})
    bin_df = pd.DataFrame(bin_rows).set_index("run")
    for n in TEST_RUNS:
        results_df.loc[n, "test_bin_accuracy"] = bin_df.loc[n, "bin accuracy"]
    display(bin_df.style.format("{:.3f}"))
else:
    print("Class names differ from the 12-class garbage dataset; bin mapping skipped.")

**Is the difference real?** Two checks on the same test images:
* **McNemar's exact test** on accuracy uses only the images where exactly one model is right.
* A **paired, class-stratified bootstrap** gives a 95% interval for the macro-F1 difference (our main metric).

The first pair is best against best. The second is the matched pair, where only the architecture differs.

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import f1_score


def compare(a, b, split="test", n_boot=2000):
    y = Y[split]
    pa, pb = PROBS[a][split].argmax(1), PROBS[b][split].argmax(1)
    n01, n10 = int(np.sum((pa == y) & (pb != y))), int(np.sum((pa != y) & (pb == y)))
    p_mc = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    mf1 = lambda t, p: f1_score(t, p, average="macro", labels=range(NUM_CLASSES), zero_division=0)
    r = np.random.default_rng(SEED)
    by_cls = [np.where(y == k)[0] for k in range(NUM_CLASSES) if np.any(y == k)]
    diffs = []
    for _ in range(n_boot):
        i = np.concatenate([r.choice(ix, len(ix)) for ix in by_cls])
        diffs.append(mf1(y[i], pb[i]) - mf1(y[i], pa[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"comparison": f"{b} vs {a}", "macro F1 difference": mf1(y, pb) - mf1(y, pa), "95% CI low": lo, "95% CI high": hi,
            f"only {a.split('-')[0]} right": n01, f"only {b.split('-')[0]} right": n10, "McNemar exact p": p_mc}


pairs_to_test = [(BEST_FCNN, BEST_CNN)] + ([MATCHED_PAIR] if MATCHED_PAIR != (BEST_FCNN, BEST_CNN) else [])
compare_df = pd.DataFrame([compare(a, b) for a, b in pairs_to_test])
compare_df.to_csv(LOGS_DIR / "fcnn_vs_cnn_tests.csv", index=False)
display(compare_df.style.format({"macro F1 difference": "{:+.3f}", "95% CI low": "{:+.3f}", "95% CI high": "{:+.3f}",
                                 "McNemar exact p": "{:.1e}"}))

**Does the model use the white background as a shortcut?** The EDA found that most battery and trash images have a white border. Two checks on the test set, for the two selected models:
1. **Recall split by background**: per class, recall on white-border images vs. the rest.
2. **Counterfactual**: paint the outer 10% of every test image grey (the training mean colour) and predict again. If battery and trash recall falls much more than the other classes, the model leaned on the white border rather than the object.

In [ ]:
def grey_border(x, frac=0.1):
    x = x.copy()
    b = int(x.shape[1] * frac)
    fill = np.round(MEAN).astype(np.uint8)
    x[:, :b], x[:, -b:], x[:, :, :b], x[:, :, -b:] = fill, fill, fill, fill
    return x


white = parts["test"]["white_border"].values > 0.6
X_grey = grey_border(X["test"])
sc_rows = []
for n in (BEST_FCNN, BEST_CNN):
    pred = PROBS[n]["test"].argmax(1)
    pred_g = predict_probs(MODELS[n], X_grey).argmax(1)
    for k, cls in enumerate(CLASS_NAMES):
        m = Y["test"] == k
        rec_of = lambda mask, p=pred: float(np.mean(p[mask] == k)) if mask.any() else np.nan
        sc_rows.append({"model": n, "class": cls, "n white": int((m & white).sum()),
                        "recall, white border": rec_of(m & white), "recall, other": rec_of(m & ~white),
                        "recall": rec_of(m), "recall, border greyed": rec_of(m, pred_g)})
shortcut = pd.DataFrame(sc_rows)
shortcut["drop when greyed"] = shortcut["recall"] - shortcut["recall, border greyed"]
shortcut.to_csv(LOGS_DIR / "shortcut_check.csv", index=False)
display(shortcut.pivot(index="class", columns="model",
                       values=["recall, white border", "recall, other", "drop when greyed"]).round(2))

suspects = [class_to_id[c] for c in ("battery", "trash") if c in class_to_id]
if suspects:
    others = ~np.isin(Y["test"], suspects)
    for n in (BEST_FCNN, BEST_CNN):
        pred = PROBS[n]["test"].argmax(1)
        a = np.isin(pred[others & white], suspects).mean() if (others & white).any() else np.nan
        b = np.isin(pred[others & ~white], suspects).mean() if (others & ~white).any() else np.nan
        print(f"{n}: other-class test images predicted battery/trash: {a:.1%} with a white border vs {b:.1%} without")

In [ ]:
def show_mistakes(name, n=12):
    pred = PROBS[name]["test"].argmax(1)
    conf = PROBS[name]["test"].max(1)
    wrong = np.where(pred != Y["test"])[0]
    wrong = wrong[np.argsort(-conf[wrong])][:n]          # most confident mistakes first
    fig, axes = plt.subplots(2, n // 2, figsize=(2.2 * n // 2, 5))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(X["test"][i])
        ax.set_title(f"true: {CLASS_NAMES[Y['test'][i]]}\npred: {CLASS_NAMES[pred[i]]} ({conf[i]:.0%})", fontsize=8, color="crimson")
    fig.suptitle(f"{name}: most confident mistakes on the test set")
    plt.tight_layout(); save_fig(fig, f"mistakes_{name}"); plt.show()


show_mistakes(BEST_CNN)
show_mistakes(BEST_FCNN)

**What did each model learn?** For 1,500 validation images we take three representations: raw pixels, the FCNN's last hidden layer and the CNN's last hidden layer. Each is reduced to 50 dimensions with PCA, then:
* a **5-nearest-neighbour classifier** (5-fold cross-validated balanced accuracy) puts a number on how well same-class images sit together,
* **t-SNE** draws a 2-D picture. Only read whether same-colour points group together; distances between clusters and cluster sizes mean nothing in t-SNE.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score


def penultimate_features(model, x):
    last_hidden = [l.name for l in model.layers if l.name.endswith("relu") or l.name.startswith("relu")][-1]
    extractor = keras.Model(model.inputs, model.get_layer(last_hidden).output)
    extractor.compile(jit_compile=False)
    return extractor.predict(make_ds(x, np.zeros(len(x), "int32"), False), verbose=0).astype("float32")


sel = np.random.default_rng(SEED).choice(len(X["val"]), min(1500, len(X["val"])), replace=False)
yv = Y["val"][sel]
spaces = {"raw pixels (32x32)": tf.image.resize(X["val"][sel].astype("float32"), (32, 32)).numpy().reshape(len(sel), -1) / 255.0,
          BEST_FCNN: penultimate_features(MODELS[BEST_FCNN], X["val"][sel]),
          BEST_CNN: penultimate_features(MODELS[BEST_CNN], X["val"][sel])}
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (title, f) in zip(axes, spaces.items()):
    f50 = PCA(min(50, f.shape[1], len(sel) - 1), random_state=SEED).fit_transform(f)
    knn = cross_val_score(KNeighborsClassifier(5), f50, yv, cv=5, scoring="balanced_accuracy").mean()
    emb = TSNE(n_components=2, perplexity=min(30, len(sel) // 4), init="pca", random_state=SEED).fit_transform(f50)
    for k, cls in enumerate(CLASS_NAMES):
        ax.scatter(emb[yv == k, 0], emb[yv == k, 1], s=6, alpha=0.7, color=cmap(k), label=cls)
    ax.set(title=f"{title}: t-SNE (validation)\n5-NN balanced accuracy = {knn:.2f}", xticks=[], yticks=[])
axes[-1].legend(markerscale=3, fontsize=7, ncol=2)
plt.tight_layout(); save_fig(fig, "tsne_features"); plt.show()

## 9. Grad-CAM: where does the CNN look?

Grad-CAM (Selvaraju et al., 2017) takes the gradient of the class score (before the softmax) with respect to the last convolutional feature maps, weights each map by that gradient and overlays the result on the image. Red areas pushed the decision most. A good model looks at the object; a model relying on a shortcut lights up the background. For battery and trash we deliberately pick white-background images, where a shortcut would show.

In [ ]:
def grad_cam(model, img_uint8, layer_name=GRADCAM_LAYER, class_idx=None):
    out = model.get_layer("class_probs")      # rebuild the pre-softmax class score from the last hidden layer
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.get_layer("head_relu").output])
    x = tf.convert_to_tensor(img_uint8[None].astype("float32"))
    with tf.GradientTape() as tape:
        fmap, h = grad_model(x, training=False)
        logits = tf.matmul(tf.cast(h, tf.float32), tf.convert_to_tensor(out.kernel)) + tf.convert_to_tensor(out.bias)
        if class_idx is None:
            class_idx = int(tf.argmax(logits[0]))
        score = logits[:, class_idx]
    grads = tf.cast(tape.gradient(score, fmap), tf.float32)
    fmap = tf.cast(fmap, tf.float32)
    cam = tf.nn.relu(tf.reduce_sum(fmap * tf.reduce_mean(grads, axis=(1, 2))[:, None, None, :], axis=-1))[0].numpy()
    return cam / (cam.max() + 1e-8), class_idx, tf.nn.softmax(logits)[0].numpy()


def overlay(img_uint8, cam, alpha=0.45):
    heat = np.asarray(Image.fromarray(np.uint8(cam * 255)).resize(img_uint8.shape[:2][::-1], Image.BILINEAR)) / 255.0
    color = plt.get_cmap("jet")(heat)[..., :3]
    return np.clip((1 - alpha) * img_uint8 / 255.0 + alpha * color, 0, 1)


# Grad-CAM runs on a float32 copy: with mixed precision the small float16 gradients can underflow to zero
keras.mixed_precision.set_global_policy("float32")
cnn_model = BUILDERS[BEST_CNN]()
cnn_model.set_weights(MODELS[BEST_CNN].get_weights())


def pick(k):
    idx = np.where(Y["test"] == k)[0]
    if CLASS_NAMES[k] in ("battery", "trash") and white[idx].any():
        idx = idx[white[idx]]
    return rng.choice(idx)


picks = [pick(k) for k in range(NUM_CLASSES)]
cols = 6
rows_n = int(np.ceil(len(picks) / cols))
fig, axes = plt.subplots(rows_n * 2, cols, figsize=(2.2 * cols, 4.6 * rows_n))
for k, i in enumerate(picks):
    cam, c, p = grad_cam(cnn_model, X["test"][i])
    r, col = 2 * (k // cols), k % cols
    axes[r, col].imshow(X["test"][i]); axes[r, col].axis("off")
    axes[r, col].set_title(f"true: {CLASS_NAMES[Y['test'][i]]}", fontsize=8)
    axes[r + 1, col].imshow(overlay(X["test"][i], cam)); axes[r + 1, col].axis("off")
    ok = c == Y["test"][i]
    axes[r + 1, col].set_title(f"pred: {CLASS_NAMES[c]} ({p[c]:.0%})", fontsize=8, color="green" if ok else "crimson")
for k in range(len(picks), rows_n * cols):          # hide unused panels
    r, col = 2 * (k // cols), k % cols
    axes[r, col].axis("off"); axes[r + 1, col].axis("off")
fig.suptitle(f"Grad-CAM for {BEST_CNN}: one test image per class "
             f"({cnn_model.get_layer(GRADCAM_LAYER).output.shape[1]}x{cnn_model.get_layer(GRADCAM_LAYER).output.shape[2]} feature map)")
plt.tight_layout(); save_fig(fig, "gradcam_test"); plt.show()
if GPUS:
    keras.mixed_precision.set_global_policy("mixed_float16")

## 10. Compute budget actually used

The lecturer asked whether we have enough computing power to train from scratch. These are the measured numbers from this run. Fewer weights does not mean less work: the CNN shares its weights across the image, so it does far more arithmetic per image than an FCNN with ten times the weights. Both numbers are reported.

In [ ]:
def model_size_mb(model):
    return sum(np.prod(w.shape) for w in model.weights) * 4 / 2**20


def forward_gflops(model):
    total = 0
    for l in model.layers:
        if isinstance(l, layers.Conv2D):
            h, w_, c = l.output.shape[1:]
            total += 2 * h * w_ * c * int(np.prod(l.kernel.shape[:-1]))
        elif isinstance(l, layers.Dense):
            total += 2 * int(np.prod(l.kernel.shape))
    return total / 1e9


def latency_ms(model, n=30):
    x = tf.convert_to_tensor(X["test"][:1].astype("float32"))
    model(x, training=False)                     # warm-up
    t0 = time.perf_counter()
    for _ in range(n):
        model(x, training=False)
    return (time.perf_counter() - t0) / n * 1000


compute_rows = []
for name, model in MODELS.items():
    rec = RESULTS[name]
    compute_rows.append({"run": name, "params": rec["params"], "GFLOP/image": forward_gflops(model),
                         "weights_MB": model_size_mb(model), "policy": rec.get("dtype_policy", "?"),
                         "epochs": rec["epochs_run"], "sec/epoch": rec["sec_per_epoch"],
                         "train_min": rec["train_minutes"], "peak_GPU_MB": rec["peak_gpu_mb"],
                         "ms/image (batch 1)": latency_ms(model)})
compute_df = pd.DataFrame(compute_rows).set_index("run")
compute_df["peak_GPU_MB"] = pd.to_numeric(compute_df["peak_GPU_MB"], errors="coerce")
compute_df.to_csv(LOGS_DIR / "compute_budget.csv")
print(f"Hardware: {GPU_NAME} | images per epoch: {len(X['train']):,} | batch size {BATCH_SIZE}")
display(compute_df.style.format({"params": "{:,}", "GFLOP/image": "{:.3f}", "weights_MB": "{:.1f}", "sec/epoch": "{:.1f}",
                                 "train_min": "{:.1f}", "peak_GPU_MB": "{:.0f}", "ms/image (batch 1)": "{:.1f}"}, na_rep="n/a"))
print(f"Total training time for all {len(MODELS)} runs: {compute_df['train_min'].sum():.0f} minutes")

fig, ax = plt.subplots(figsize=(8, 4.6))
for name in results_df.index:
    fam = results_df.loc[name, "family"]
    xv, yv_ = results_df.loc[name, "params"], results_df.loc[name, "val_macro_f1"]
    ax.scatter(xv, yv_, s=60, color="#E45756" if fam == "FCNN" else "#4C78A8")
    ax.annotate(f"{name} ({compute_df.loc[name, 'GFLOP/image']:.2f} GFLOP)", (xv, yv_),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set(xscale="log", xlabel="weights (log scale)", ylabel="validation macro F1",
       title="Validation macro F1 against number of weights (red = FCNN, blue = CNN)")
plt.tight_layout(); save_fig(fig, "params_vs_f1"); plt.show()

## 11. Export the best models for the real-life photo demo

Two settings for the demo are fixed here on **validation** data, never on our own photos:
* **Test-time augmentation (TTA)**: averaging the prediction over the photo, its mirror image and a tighter crop. Kept only if it raises validation macro F1.
* **"Not sure" threshold**: the lowest CNN confidence at which its answers are right at least 90% of the time on validation. Below it the demo says "not sure".

The two winning models are then rebuilt in plain float32 (no mixed precision) and saved as `.keras` files without optimiser state, so they are small and run on any machine. Open **`02_real_life_photo_demo.ipynb`** next.

In [ ]:
def tight_crop(x):
    m = IMG_SIZE // 10
    return np.stack([np.asarray(Image.fromarray(a[m:-m, m:-m]).resize((IMG_SIZE, IMG_SIZE), RESAMPLE)) for a in x])


def tta_probs(model, x):
    views = (x, np.ascontiguousarray(x[:, :, ::-1]), tight_crop(x))
    return np.mean([predict_probs(model, v) for v in views], axis=0)


use_tta = {}
for n in (BEST_FCNN, BEST_CNN):
    base_f1 = scores(Y["val"], PROBS[n]["val"].argmax(1))["macro_f1"]
    tta_f1 = scores(Y["val"], tta_probs(MODELS[n], X["val"]).argmax(1))["macro_f1"]
    use_tta[n] = bool(tta_f1 > base_f1 + 0.005)
    print(f"{n}: validation macro F1 {base_f1:.3f} -> {tta_f1:.3f} with TTA -> use TTA: {use_tta[n]}")

conf_v = PROBS[BEST_CNN]["val"].max(1)
ok_v = PROBS[BEST_CNN]["val"].argmax(1) == Y["val"]
grid = np.round(np.arange(0.30, 0.91, 0.01), 2)
sel_acc = np.array([ok_v[conf_v >= t].mean() if (conf_v >= t).any() else np.nan for t in grid])
hits = np.where(sel_acc >= 0.90)[0]
UNSURE_BELOW = float(grid[hits[0]]) if len(hits) else 0.5
kept = conf_v >= UNSURE_BELOW
print(f"'Not sure' below {UNSURE_BELOW:.2f}: on validation the CNN answers {kept.mean():.0%} of images "
      f"and is right on {ok_v[kept].mean() if kept.any() else float('nan'):.0%} of those")

In [ ]:
keras.mixed_precision.set_global_policy("float32")
export = {}
for tag, name in (("fcnn", BEST_FCNN), ("cnn", BEST_CNN)):
    clean = BUILDERS[name]()
    clean.set_weights(MODELS[name].get_weights())
    path = EXPORT_DIR / f"best_{tag}.keras"
    clean.save(path)
    export[tag] = {"run": name, "file": path.name, "use_tta": use_tta[name],
                   **{f"test_{k}": float(results_df.loc[name, f"test_{k}"])
                      for k in ("accuracy", "balanced_acc", "macro_f1", "bin_accuracy") if f"test_{k}" in results_df}}
    print(f"Saved {name} -> {path} ({path.stat().st_size / 2**20:.1f} MB)")

demo_meta = {
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "preprocessing": "EXIF rotation fix -> RGB (transparent = white) -> centre square crop -> resize to img_size (Lanczos); "
                     "pixel values stay 0-255, the model standardises internally",
    "gradcam_layer": GRADCAM_LAYER,
    "unsure_below": UNSURE_BELOW,
    "bins": {c: BIN[c] for c in CLASS_NAMES if c in BIN},
    "models": export,
    "dataset": KAGGLE_DATASET,
    "quick_run": QUICK_RUN,
}
(EXPORT_DIR / "meta.json").write_text(json.dumps(demo_meta, indent=2))
results_df.to_csv(LOGS_DIR / "results_all_runs.csv")
archive = shutil.make_archive(str(OUT_DIR / "waste_models_export"), "zip", EXPORT_DIR)
print("Export folder:", EXPORT_DIR)
print("Zipped copy  :", archive)
display(pd.DataFrame(export).T)

## 12. Optional: how much do results change with the random seed?

Each configuration above was trained once. A small difference between two runs could be luck. Here the cheaper configurations are trained again with two more seeds, and we report the mean and standard deviation of validation macro F1. A design difference counts as real only if it is clearly larger than about two standard deviations. The extra models are not used for model selection or the demo. Set `RUN_EXTRA_SEEDS = False` in Part 1 to skip this part.

In [ ]:
SEED_CONFIGS = ("FCNN-A", "FCNN-B", "FCNN-B32", "FCNN-C", "CNN-B64")
if RUN_EXTRA_SEEDS:
    for base in SEED_CONFIGS:
        epochs_cap = MAX_EPOCHS_CNN if base.startswith("CNN") else MAX_EPOCHS_FCNN
        for s in (SEED + 1, SEED + 2):
            run_experiment(f"{base}_s{s}", BUILDERS[base], epochs_cap, seed=s)
    seed_df = pd.DataFrame([{"config": k.split("_s")[0], "seed": r.get("seed", SEED),
                             "val_macro_f1": r.get("val_macro_f1", max(r["history"]["val_macro_f1"]))}
                            for k, r in RESULTS.items() if k.split("_s")[0] in SEED_CONFIGS])
    seed_summary = seed_df.groupby("config")["val_macro_f1"].agg(["mean", "std", "count"])
    seed_summary.to_csv(LOGS_DIR / "seed_variation.csv")
    display(seed_summary.style.format({"mean": "{:.3f}", "std": "{:.3f}"}))
else:
    print("Skipped (RUN_EXTRA_SEEDS = False).")

### What to take into the report
* Figures are in `figures/` (class counts, samples, background bias, PCA, learning curves, confusion matrices, Grad-CAM, t-SNE, parameters vs F1).
* Tables are in `logs/`: `data_audit.csv`, `results_all_runs.csv`, `design_choices_validation.csv`, `per_class_best_models.csv`, `fcnn_vs_cnn_tests.csv`, `shortcut_check.csv`, `compute_budget.csv`, `seed_variation.csv`.
* Next step: the real-life photo test in `02_real_life_photo_demo.ipynb`.